# From your MCMC sampler to a profile-likelihood code

This notebook is a **direct continuation** of `write_MH_MCMC.ipynb`. There you built a Metropolis-Hastings sampler. Here you will turn that *same* machinery into a **global optimiser** (simulated annealing), and then wrap it into a loop that computes **profile likelihoods** — the core of [Procoli](https://github.com/tkarwal/procoli).

The whole notebook is one idea made concrete:

> The same Metropolis loop, plus a *temperature* knob and a *cooling schedule*, becomes an optimiser. Wrap that optimiser to loop over fixed values of one parameter and you have a profile-likelihood code.

You will keep editing functions you already wrote rather than starting from scratch:

| Notebook 1 (you wrote) | Notebook 2 (you adapt it to) | Change |
|---|---|---|
| `acceptance_probability(lp_new, lp_cur)` | `acceptance_probability(lp_new, lp_cur, temperature)` | divide the log-ratio by `T` — one line |
| `mh_step(θ, log_post_fn, step)` | `annealed_step(θ, log_post_fn, step, temperature)` | pass `T` through; also return the log-post value |
| `run_mh_chain(..., step_size, n_steps)` | `run_annealing(..., ladder)` | walk a ladder of `(T, δ)` rungs; shrink the step; keep the **best** point |
| — *(new)* | `profile_likelihood(...)` | fix one parameter, optimise the rest, repeat |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
from scipy import optimize
from scipy.special import logsumexp
import corner
from tqdm.auto import tqdm

# Optional interactivity (Section 2 and the optional bimodal demo).
# Leave False for a fully static, portable notebook; set True in a live
# Jupyter kernel to enable temperature / cooling-rate sliders (needs ipywidgets).
USE_WIDGETS = False

np.random.seed(42)

mpl.rcParams['figure.dpi']      = 110
mpl.rcParams['axes.labelsize']  = 12
mpl.rcParams['font.size']       = 11
mpl.rcParams['legend.fontsize'] = 10
mpl.rcParams['axes.spines.top']   = False
mpl.rcParams['axes.spines.right'] = False

---
## Section 0 — A quick recap, and a change of language

In notebook 1 we sampled a **posterior** $P(\theta\mid D) \propto \pi(\theta) \, \mathcal{L}(D\mid\theta)$. With flat priors, the posterior is proportional to the **likelihood**, so the function you sampled *was* effectively the likelihood.

Optimisation and profiling are **frequentist** ideas: they care about the **likelihood** $\mathcal{L}$, not the posterior. So from here on we will be careful about the distinction, and we will measure fit quality with the chi-squared,

$$\chi^2 \equiv -2\ln\mathcal{L}.$$

A higher log-likelihood means a *lower* $\chi^2$ (a better fit). Differences are what matter: $\Delta\chi^2 = -2\,\Delta\ln\mathcal{L}$.

> **Your functions stay in the maximise / `log_post_fn` convention from notebook 1.** We only convert to $\chi^2$ for plotting and for reading off confidence intervals.

The cell below re-provides the functions you built in notebook 1, so this notebook stands alone.

In [ ]:
# --- PROVIDED: the functions you built in notebook 1 ---
# Re-provided so this notebook runs on its own. We will *modify* these below.

def propose(theta_current, step_size):
    """Gaussian proposal centred on the current position."""
    if np.ndim(step_size) == 2:
        return np.random.multivariate_normal(mean=theta_current, cov=step_size)
    else: 
        return np.random.normal(loc=theta_current, scale=step_size) 

def acceptance_probability(log_post_proposed, log_post_current):
    """log MH acceptance probability: min(0, lp_new - lp_cur)."""
    return np.minimum(0.0, log_post_proposed - log_post_current)

def mh_step(theta_current, log_post_fn, step_size, **kwargs):
    """One Metropolis-Hastings step."""
    theta_proposed = propose(theta_current, step_size)
    lp_current  = log_post_fn(theta_current,  **kwargs)
    lp_proposed = log_post_fn(theta_proposed, **kwargs)
    log_alpha = acceptance_probability(lp_proposed, lp_current)
    if np.log(np.random.uniform()) < log_alpha:
        return theta_proposed, True
    return theta_current.copy(), False

def run_mh_chain(log_post_fn, theta_init, step_size, n_steps, progress=False, **kwargs):
    """Your notebook-1 sampler (used here to draw posteriors for comparison)."""
    theta_init = np.atleast_1d(np.asarray(theta_init, dtype=float))
    chain = np.zeros((n_steps, len(theta_init)))
    chain[0] = theta_init
    n_accepted = 0
    iterator = range(1, n_steps)
    if progress:
        iterator = tqdm(iterator, desc='MCMC', unit='step')
    for i in iterator:
        chain[i], accepted = mh_step(chain[i-1], log_post_fn, step_size, **kwargs)
        n_accepted += int(accepted)
    return chain, n_accepted / n_steps

In [ ]:
# --- PROVIDED: the toy likelihood (the 2D correlated Gaussian from notebook 1) ---
# Same function as before, now read as a LOG-LIKELIHOOD.

MU_TOY  = np.array([3.0, -1.0])
COV_TOY = np.array([[1.0, 1.6],
                    [1.6, 4.0]])          # sigma1=1, sigma2=2, rho=0.8
_COV_INV  = np.linalg.inv(COV_TOY)
_LOG_NORM = -0.5 * np.log(np.linalg.det(2 * np.pi * COV_TOY))

def log_likelihood_toy(theta):
    """Log-likelihood of the 2D correlated Gaussian."""
    d = np.asarray(theta, dtype=float) - MU_TOY
    return float(_LOG_NORM - 0.5 * d @ _COV_INV @ d)

# Per-parameter width: a sensible proposal step is delta * sqrt(Sigma_jj).
COV_SCALE_TOY = np.sqrt(np.diag(COV_TOY))     # -> [1.0, 2.0]
PARAM_LABELS  = [r'$\theta_1$', r'$\theta_2$']

---
## Section 1 — Optimisation is not sampling

Sampling and optimisation answer different questions:

- **Sampling (notebook 1)** maps out *where the probability is* — it returns a cloud of points whose density is the posterior.
- **Optimisation (today)** finds *the single best point* — the parameters that maximise the likelihood (the "best fit"), and the shape of $\mathcal{L}$ right around it.

Frequentist constraints come from that **shape near the maximum**, not from posterior mass. The figure below contrasts the two.

In [ ]:
# --- PROVIDED: sampling (a cloud) vs optimising (a path to the peak) ---

# grid to get likelihood contours
t1 = np.linspace(-1.5, 7.5, 200); t2 = np.linspace(-8.0, 6.0, 200)
T1, T2 = np.meshgrid(t1, t2)
Z = np.vectorize(lambda a, b: log_likelihood_toy([a, b]))(T1, T2)
levels = Z.max() - np.array([4.5, 2.0, 0.5])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

# left: a posterior cloud (drawn directly from the Gaussian for illustration)
cloud = np.random.multivariate_normal(MU_TOY, COV_TOY, 2000)
ax[0].contour(T1, T2, Z, levels=levels, colors='0.7', linewidths=1)
ax[0].plot(cloud[:, 0], cloud[:, 1], '.', ms=2, color='steelblue', alpha=0.3)
ax[0].set_title('Sampling: where is the probability?')

# right: a single optimisation path climbing to the peak (gradient ascent, for show)
theta = np.array([-1.0, -6.0]); path = [theta.copy()]
for _ in range(40):
    theta = theta + 0.4 * (-_COV_INV @ (theta - MU_TOY))
    path.append(theta.copy())
path = np.array(path)
ax[1].contour(T1, T2, Z, levels=levels, colors='0.7', linewidths=1)
ax[1].plot(path[:, 0], path[:, 1], '-o', ms=3, color='darkorange')
ax[1].plot(*MU_TOY, '*', color='gold', ms=12, mec='k', zorder=6, alpha=0.5)
ax[1].set_title('Optimising: where is the single best fit?')

for a in ax:
    a.set_xlabel(PARAM_LABELS[0])
ax[0].set_ylabel(PARAM_LABELS[1])
plt.tight_layout(); plt.show()

---
## Section 2 — The temperature knob

To turn the sampler into an optimiser we add **one knob**: a temperature $T$. Procoli accepts a move with probability

$$\alpha = \min\!\left(1,\ \left(\frac{\mathcal{L}_{\rm new}}{\mathcal{L}_{\rm old}}\right)^{1/T}\right)
\qquad\Longleftrightarrow\qquad
\log\alpha = \min\!\left(0,\ \frac{\ln\mathcal{L}_{\rm new}-\ln\mathcal{L}_{\rm old}}{T}\right).$$

Think of it like a **gas**. A *hot* gas has high mean molecular speed and a long mean free path: molecules jitter, range widely, and bounce all over the container — they **explore**. As you **cool** the gas, the molecules slow down, mean free paths shorten, and they settle into low-energy niches — they **hone in**. (The name *annealing* comes from metallurgy: cool a metal slowly and its atoms settle into a low-energy crystal; cool it too fast and defects freeze in. We will see that trap in the optional Section 10.)

For our optimiser:

- $T > 1$ **flattens** the landscape → almost everything is accepted → wide exploration.
- $T = 1$ is ordinary Metropolis-Hastings (notebook 1).
- $T < 1$ **sharpens** the landscape → only good moves survive → honing in.
- $T \to 0$ accepts *only* improvements → a greedy hill-climber.

The figure shows a 1-D slice of the likelihood raised to $1/T$: as $T$ falls, the distribution sharpens to a spike at the maximum.

In [ ]:
# --- PROVIDED: how temperature sharpens the landscape ---

def tempered_slice(x, temperature):
    """Likelihood along theta_1 (at theta_2 = -1), raised to 1/T, peak-normalised."""
    logL = np.array([log_likelihood_toy([xi, -1.0]) for xi in x])
    g = np.exp((logL - logL.max()) / temperature)
    return g 

xs = np.linspace(-1, 7, 300)

def plot_tempering(temps=(5.0, 1.0, 0.2, 0.01)):
    fig, ax = plt.subplots(figsize=(7, 4))
    for T, col in zip(temps, ['firebrick', 'darkorange', 'seagreen', 'steelblue']):
        ax.plot(xs, tempered_slice(xs, T), color=col, lw=2, label=f'T = {T:g}')
    ax.axvline(3.0, color='k', ls=':', lw=1, label='maximum')
    ax.set_xlabel(PARAM_LABELS[0]); ax.set_ylabel('$\mathcal{L}^{1/T}$ (normalised)')
    ax.set_title('Cooling sharpens the landscape'); ax.legend(frameon=False)
    plt.tight_layout(); plt.show()

if USE_WIDGETS:
    from ipywidgets import interact, FloatLogSlider
    interact(lambda T: plot_tempering((T,)),
             T=FloatLogSlider(value=1.0, base=10, min=-2, max=1.7, description='T'))
else:
    plot_tempering()

---
## Section 3 — Turn your sampler into an optimiser

Two small edits to functions you already have.

> **Tip:** as in notebook 1, each body is only a few lines. Keep it minimal.

In [ ]:
# --- STUDENT (3a): add a temperature to the acceptance probability ---
# This REPLACES your notebook-1 acceptance_probability. The default temperature=1.0
# means every call from notebook 1 still behaves exactly as before.

def acceptance_probability(log_post_proposed, log_post_current, temperature=1.0):
    """
    log Metropolis acceptance probability WITH a temperature.

    Procoli accepts with probability (L_new / L_old)^(1/T); in log space that is
    (lp_new - lp_cur) / T, capped at 0.

    Parameters
    ----------
    log_post_proposed, log_post_current : float
    temperature : float
        T = 1 recovers ordinary Metropolis-Hastings.

    Returns
    -------
    log_alpha : float
        min(0, (log_post_proposed - log_post_current) / temperature)
    """
    pass   # 1 line: like notebook 1, but divide the difference by temperature

In [ ]:
# --- STUDENT (3b): one annealed step ---

def annealed_step(theta_current, log_post_fn, step_size, temperature, **kwargs):
    """
    One Metropolis step at a given temperature.

    Like your mh_step, with two changes:
      (i)  pass `temperature` to acceptance_probability;
      (ii) ALSO return the log-posterior of the returned position, so the runner
           can track the best fit without recomputing it.

    Parameters
    ----------
    theta_current : np.ndarray
    log_post_fn : callable
    step_size : float or np.ndarray
    temperature : float
    **kwargs : passed to log_post_fn

    Returns
    -------
    theta_next : np.ndarray
    log_post_next : float
        log_post_fn evaluated at theta_next.
    accepted : bool
    """
    pass   # ~6 lines:
           # 1. propose theta_proposed
           # 2. evaluate log_post_fn at current and proposed
           # 3. log_alpha = acceptance_probability(.., .., temperature)
           # 4. draw u ~ Uniform(0,1)
           # 5. if accepted return (proposed, its log-post, True)
           # 6. else        return (current,  its log-post, False)

---
## Section 4 — The cooling ladder and the master optimiser

A simulated-annealing run walks down a **ladder** of rungs. Each rung has a temperature `T`, a jump factor `δ`, and a number of steps. Temperature and step size shrink together: start hot and wide to explore, end cold and narrow to settle. The proposal width on each rung is

$$\Delta\theta_j \simeq \delta\,\sqrt{\Sigma_{jj}},$$

i.e. `δ` times each parameter's own scale (`cov_scale`), so every parameter is stepped sensibly regardless of its units.

Now write the master optimiser. Most of the bookkeeping is provided — you fill the **three-line core** that takes a step, tracks the best fit, and records the trajectory.

In [ ]:
# --- PROVIDED: a cooling ladder ---
# Each rung is (temperature, jump factor delta, number of steps).
LADDER = [
    (50.0, 2.00, 400),
    (10.0, 1.50, 400),
    ( 3.0, 1.00, 400),
    ( 1.0, 0.60, 400),
    ( 0.3, 0.30, 400),
    ( 0.1, 0.15, 400),
    (0.03, 0.08, 600),
]

In [ ]:
# --- STUDENT: the master optimiser ---

def run_annealing(log_post_fn, theta_init, ladder, cov_scale=None,
                  progress=False, **kwargs):
    """
    Anneal from theta_init down the cooling ladder; return the best fit found.

    Parameters
    ----------
    log_post_fn : callable
        Function to MAXIMISE (a log-likelihood, or a log-posterior).
    theta_init : array-like
    ladder : list of (temperature, delta, n_steps)
    cov_scale : np.ndarray or None
        Per-parameter width; step size = delta * cov_scale. None -> ones.
    progress : bool
    **kwargs : passed through to log_post_fn

    Returns
    -------
    result : dict with keys
        'theta_best'  : best-fit parameter vector
        'chi2_best'   : -2 * (best log-posterior)
        'trajectory'  : (n_total, n_params) every visited position
        'chi2_trace'  : (n_total,) -2 * log-posterior at each step
        'rungs'       : list of (start_step, end_step, T, delta)
    """
    theta = np.atleast_1d(np.asarray(theta_init, dtype=float))
    if cov_scale is None:
        cov_scale = np.ones(len(theta))

    logpost = log_post_fn(theta, **kwargs)
    theta_best, logpost_best = theta.copy(), logpost

    trajectory    = [theta.copy()]
    logpost_trace = [logpost]
    rungs = []
    step  = 0

    rung_iter = tqdm(ladder, desc='annealing', unit='rung') if progress else ladder
    for (temperature, delta, n_steps) in rung_iter:
        step_size = delta * cov_scale          # Delta theta_j ~ delta * sqrt(Sigma_jj)
        start = step
        for _ in range(n_steps):
            # ---------------- TODO (your three-line core) ----------------
            # 1. take one annealed_step at this temperature; unpack
            #    (theta, logpost, accepted)
            # 2. if logpost beats logpost_best, update theta_best & logpost_best
            # 3. append theta.copy() to trajectory and logpost to logpost_trace
            pass
            # -------------------------------------------------------------
            step += 1
        rungs.append((start, step, temperature, delta))

    return dict(theta_best=theta_best,
                chi2_best=-2.0 * logpost_best,
                trajectory=np.array(trajectory),
                chi2_trace=-2.0 * np.array(logpost_trace),
                rungs=rungs)

In [ ]:
# --- PROVIDED: optimise the toy and check the best fit ---

np.random.seed(0)
result = run_annealing(log_likelihood_toy, theta_init=np.array([0.0, 0.0]),
                       ladder=LADDER, cov_scale=COV_SCALE_TOY, progress=True)

print(f"Best-fit theta : {result['theta_best'].round(3)}   (truth {MU_TOY})")
print(f"chi2_best      : {result['chi2_best']:.3f}   (analytic {-2*_LOG_NORM:.3f})")
print(f"Total steps    : {len(result['trajectory'])}")

---
## Section 5 — Watch it cool

Two diagnostics. 

First, $\Delta\chi^2$ (relative to the best fit) versus step, coloured by ladder rung — the optimiser's equivalent of Figure 2 in the [Procoli paper](https://arxiv.org/abs/2401.14225). The hot rungs roam over large $\Delta\chi^2$; the cold rungs settle towards zero. 

In [ ]:
# --- PROVIDED: chi-squared vs step, coloured by cooling rung ---

def plot_cooling(result, chi2_min=None, ax=None):
    if chi2_min is None:
        chi2_min = result['chi2_trace'].min()
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 4))
    dchi2 = np.maximum(result['chi2_trace'] - chi2_min, 1e-3)
    cmap = plt.cm.viridis(np.linspace(0, 0.88, len(result['rungs'])))
    for (s, e, T, d), col in zip(result['rungs'], cmap):
        ax.plot(np.arange(s, e), dchi2[s:e], color=col, lw=0.8,
                label=f'T={T:g}, δ={d:g}')
    ax.axhline(1e-3, color='k', ls='--', lw=1)
    ax.set_yscale('log')
    ax.set_xlabel('MCMC step'); ax.set_ylabel(r'$\Delta\chi^2$ from best fit')
    ax.set_title('Cooling trajectory (cf. Procoli Fig. 2)')
    ax.legend(fontsize=7, ncol=2, loc='upper right')
    return ax

plot_cooling(result, chi2_min=-2*_LOG_NORM)
plt.tight_layout(); plt.show()

Second, the path itself in parameter space, with the steps shrinking as the chain cools. 

This is worth exploring interactively to understand what's going on: pan around, and zoom into the tight cluster near the peak to watch the rungs hone in as they cool. 

In [ ]:
# --- PROVIDED: the optimisation path in parameter space ---

# Set USE_WIDGETS = True at the top of the notebook (the same flag as the
# Section 2 sliders) to make this figure interactive. That needs ipympl:
#   conda install -c conda-forge ipympl     (then restart the kernel)
# If USE_WIDGETS is False, or ipympl is missing, you get a normal static plot.
#
# Note: %matplotlib sets the backend for the WHOLE kernel, so later plots will
# also be interactive. Run `%matplotlib inline` to return to static.

if USE_WIDGETS:
    try:
        import ipympl                                   # noqa: F401
        get_ipython().run_line_magic('matplotlib', 'widget')
    except ModuleNotFoundError:
        get_ipython().run_line_magic('matplotlib', 'inline')
        print('ipympl not found -> static plot. For pan/zoom: '
              'conda install -c conda-forge ipympl, then restart the kernel.')

def plot_trajectory_2d(result, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    ax.contour(T1, T2, Z, levels=levels, colors='0.75', linewidths=1)
    traj = result['trajectory']
    cmap = plt.cm.viridis(np.linspace(0, 0.88, len(result['rungs'])))
    for (s, e, T, d), col in zip(result['rungs'], cmap):
        ms = np.interp(T, [0.03, 50], [2.5, 9.0])
        ax.plot(traj[s:e, 0], traj[s:e, 1], '.', color=col, ms=ms, alpha=0.35)
    ax.plot(*result['theta_best'], '*', color='gold', ms=18, mec='k',
            zorder=6, label='best fit')
    ax.plot(*MU_TOY, 'x', color='red', ms=10, mew=2, zorder=6, label='truth')
    ax.set_xlabel(PARAM_LABELS[0]); ax.set_ylabel(PARAM_LABELS[1])
    ax.set_title('Cooling: wide jumps when hot, small when cold')
    ax.legend(loc='upper left', frameon=False)
    return ax

plot_trajectory_2d(result)
plt.tight_layout(); plt.show()

Your hand-written annealer should land where `scipy`'s optimiser does. For a simple case like this one, `scipy` might beat your optimiser. However, for complicated, higher-dimensional, noisy cosmological likelihoods, `scipy` often fails to find the true minimum and simulated-annealing performs better. 

In [ ]:
# --- PROVIDED (optional): sanity-check against a library optimiser ---

ref = optimize.minimize(lambda th: -log_likelihood_toy(th),
                        x0=[0.0, 0.0], method='Nelder-Mead')
print("scipy Nelder-Mead best :", ref.x.round(3))
print("your annealing best    :", result['theta_best'].round(3))

---
## Section 6 — Cooling too fast: getting stuck in a local minimum

*We may not get to this in class, but it's here for after class if you're interested.*

Our toys so far had a single optimum. Real likelihoods can have several. 

Then, **how fast you cool matters**. 

Cool too quickly from a far-off point and the optimiser freezes in whichever basin it began in; but spend time at a hot-temperature rung first and it can cross the barrier to the true (global) minimum. This is exactly the metallurgical intuition behind the name "annealing."

In [ ]:
# --- PROVIDED (OPTIONAL): a bimodal likelihood and two cooling schedules ---
M1, S1, A1 = -2.0, 0.6, 1.0      # shallower (local) well
M2, S2, A2 =  2.5, 0.6, 2.0      # deeper (global) well -> taller likelihood

def log_like_bimodal(theta):
    x = float(theta[0])
    return float(logsumexp([np.log(A1) - 0.5*((x - M1)/S1)**2,
                            np.log(A2) - 0.5*((x - M2)/S2)**2]))

FAST = [(0.05, 0.15, 2800)]                                   # straight to cold temperature
SLOW = [(30.0, 1.5, 500), (5.0, 1.0, 500), (1.0, 0.6, 500),
        (0.2, 0.3, 500), (0.05, 0.1, 800)]                    # cool gradually, same total steps

start = np.array([-3.0])                                      # try varying starting positions -  
                                                              # start in the shallow and deeper wells
# run an optimiser, we only seek the global peak, not a profile 
np.random.seed(10)
fast = run_annealing(log_like_bimodal, start, FAST, cov_scale=np.array([1.0]), progress=True)
np.random.seed(10)
slow = run_annealing(log_like_bimodal, start, SLOW, cov_scale=np.array([1.0]), progress=True)

xg = np.linspace(-5, 6, 400)
Lg = np.array([np.exp(log_like_bimodal([x])) for x in xg])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xg, Lg, color='0.6', lw=2)
ax.axvline(start[0], color='k', ls=':', lw=1, label='start')
ax.plot(fast['theta_best'][0], np.exp(log_like_bimodal(fast['theta_best'])), 'v',
        color='firebrick', ms=8, label=f"fast cool -> x={fast['theta_best'][0]:.2f}")
ax.plot(slow['theta_best'][0], np.exp(log_like_bimodal(slow['theta_best'])), '*',
        color='seagreen', ms=12, label=f"slow cool -> x={slow['theta_best'][0]:.2f} (global)")
ax.set_xlabel('x'); ax.set_ylabel('likelihood')
ax.set_title('Cool too fast and you can freeze in the wrong basin'); ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

---
## Section 7 — Sequential profile likelihoods

A **profile likelihood** for a parameter $\theta_i$ asks: 

*For each fixed value of $\theta_i$, how well can the model fit the data if every other parameter is free to adjust?* 

The recipe:

1. choose a grid of values for the parameter of interest $\theta_i$
2. **fix** $\theta_i$ at a grid value
3. **optimise** all the other (nuisance) parameters — that is one `run_annealing` call
4. record the best-fit $\chi^2$ at that grid value
5. move to the next grid value and repeat

The curve $\chi^2_{\min}(\theta_i)$ is the profile. Confidence intervals are read where $\Delta\chi^2 = 1$ ($1\sigma$) , $\Delta\chi^2 = 4$ ($2\sigma$) and so on. 

**Why "sequential"?** Neighbouring grid points have almost the same best-fit nuisance parameters, so we **start each optimisation from the previous point's best fit**: the optimiser begins right next to the answer and converges fast. 

If the first optimisation seeks aglobal minimum, then subsequent optimisations can be less rigirous and still find their minima. This sequential profiling approach is used in Procoli to speed up the profiler. 

But here we simply set the grid to serach on for the profiled parameter. 

In [ ]:
# --- PROVIDED: fix one parameter, leave the rest free ---

def clamp_param(log_fn, fixed_index, fixed_value):
    """
    Return a function of the REMAINING parameters, with one parameter fixed.

    The returned function takes the reduced parameter vector (all params except
    `fixed_index`) and inserts `fixed_value` back in before calling log_fn.

    Note: this is a *factory* -- it returns a function (`reduced_fn`); it does
    not evaluate anything itself. `theta_reduced` and any `**kwargs` are inputs
    to `reduced_fn`, supplied later each time the optimiser evaluates it.
    """
    def reduced_fn(theta_reduced, **kwargs):
        theta_full = np.insert(np.asarray(theta_reduced, dtype=float),
                               fixed_index, fixed_value)
        return log_fn(theta_full, **kwargs)
    return reduced_fn

In [ ]:
# --- STUDENT: sequential profile likelihood ---

def profile_likelihood(log_like_fn, param_index, param_grid, theta_init_full,
                       ladder, cov_scale, **kwargs):
    """
    Profile `param_index`. For each value on param_grid: fix that parameter,
    optimise the rest, and record the best-fit chi^2. Start each optimisation
    from the previous point's best fit (warm start).

    Parameters
    ----------
    log_like_fn : callable     (the LIKELIHOOD -- profiling is frequentist)
    param_index : int
    param_grid : np.ndarray
    theta_init_full : np.ndarray   (full starting guess, all parameters)
    ladder, cov_scale : as for run_annealing
    **kwargs : passed to log_like_fn (e.g. data=...)

    Returns
    -------
    chi2_profile : (n_grid,)        best-fit chi^2 at each grid value
    theta_profile : (n_grid, n_params)  full best-fit at each grid value
    """
    n_params      = len(theta_init_full)
    reduced_scale = np.delete(cov_scale, param_index)

    # a global fit gives a good first starting point (and the overall minimum)
    glob = run_annealing(log_like_fn, theta_init_full, ladder,
                         cov_scale=cov_scale, **kwargs)
    start_reduced = np.delete(glob['theta_best'], param_index)

    chi2_profile  = np.full(len(param_grid), np.nan)
    theta_profile = np.full((len(param_grid), n_params), np.nan)

    for i, value in enumerate(tqdm(param_grid, desc='profile', unit='pt')):
        # ---------------- TODO (your core, ~5 lines) ----------------
        # 1. Fix the profiled parameter at `value`: build a reduced log-likelihood
        #    over the OTHER parameters. (clamp_param is exactly this tool.)
        # 2. Optimise those remaining parameters with run_annealing -- start the
        #    search at start_reduced, step with reduced_scale, and pass **kwargs
        #    through so anything like data reaches the likelihood.
        # 3. Read the best-fit chi-squared off that optimisation's result and
        #    store it in chi2_profile[i].
        # 4. Rebuild the FULL best-fit vector by putting `value` back in at
        #    param_index, and store it in theta_profile[i]. Use np.insert to do this.
        # 5. Start the next grid point from this new point: update start_reduced to 
        #    this point's best-fit (reduced) parameters, so the next fit begins by 
        #    its answer.
        pass
        # ------------------------------------------------------------

    return chi2_profile, theta_profile

---
## Section 8 — Example 1: a flat prior, where profile and posterior agree

We profile $\theta_1$ of the toy Gaussian (optimising over $\theta_2$), and compare to the **posterior** $\theta_1$ from your notebook-1 MCMC. With flat priors the posterior equals the likelihood, so all three of {profile likelihood, MCMC posterior, analytic $\mathcal{N}(\mu_1,\Sigma_{11})$} should coincide. This is the sanity check.

> Parameters are repeated in each run cell on purpose — tweak them here and re-run without scrolling away.

In [ ]:
# --- PROVIDED: Example 1 -- profile vs posterior with a flat prior ---
theta1_grid = np.linspace(0.5, 5.5, 21)     # grid for the profiled parameter
theta_init  = np.array([0.0, 0.0])

# (a) PROFILE: optimise the LIKELIHOOD with theta_1 fixed on the grid
np.random.seed(1)
chi2_prof, theta_prof = profile_likelihood(
    log_likelihood_toy, 0, # index of profiled parameter 
    theta1_grid, theta_init, # grid and starting point defined above 
    LADDER, COV_SCALE_TOY)
dchi2 = chi2_prof - np.nanmin(chi2_prof) # normalise the profile to the minimum chi^2


# (b) POSTERIOR: sample with your nb1 MCMC (flat prior => posterior = likelihood)
np.random.seed(2)
chain1, acc1 = run_mh_chain(log_likelihood_toy, theta_init, step_size=1.5, n_steps=20000)
post1 = chain1[4000:, 0] # remove burn in 

xs_fine = np.linspace(0.5, 5.5, 300) # grid for plotting the known analytic marginal 
analytic = np.exp(-0.5 * (xs_fine - MU_TOY[0])**2 / COV_TOY[0, 0])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
# plot profile as likelihood (exp(-dchi2/2)) (this can also be normalised to the posterior)
ax[0].plot(theta1_grid, np.exp(-dchi2/2), 'o-', color='steelblue', label='profile likelihood')
# plot MCMC posterior 
ax[0].hist(post1, bins=40, density=True, color='0.8', label='MCMC posterior')
# plot the known analytic marginal 
ax[0].plot(xs_fine, analytic/np.trapz(analytic, xs_fine), 'r--', label=r'analytic $\mathcal{N}(\mu_1,\Sigma_{11})$')
# plot the true value
ax[0].axvline(MU_TOY[0], color='k', ls=':', lw=1)
# 
ax[0].set_xlabel(PARAM_LABELS[0]); ax[0].set_ylabel('density (peak-scaled)')
ax[0].set_title('Compare MCMC to profile'); ax[0].legend(fontsize=8, frameon=False)

# plot the profile as delta chi^2, with 1- and 2-sigma levels
ax[1].plot(theta1_grid, dchi2, 'o-', color='steelblue')
for lvl, lab in [(1, r'$1\sigma$'), (4, r'$2\sigma$')]:
    ax[1].axhline(lvl, color='firebrick', ls=':', lw=1)
    ax[1].text(5.4, lvl+0.05, lab, color='firebrick', ha='right', fontsize=9)
# plot the true value
ax[1].axvline(MU_TOY[0], color='k', ls=':', lw=1, label='truth')
# 
ax[1].set_xlabel(PARAM_LABELS[0]); ax[1].set_ylabel(r'$\Delta\chi^2$')
ax[1].set_title('Profile $\\Delta\\chi^2$ and confidence levels'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# profile 1-sigma limits: interpolate where Delta chi^2 crosses 1.
# dchi2 is monotonic on each side of the minimum, so on each branch we read off the
# crossing by interpolating theta_1 as a function of dchi2 (np.interp needs an
# increasing x, hence the [::-1] reversal on the left branch).
imin    = int(np.nanargmin(dchi2))
# left crossing - interpolate theta(chi2) at dchi2=1.0 
lo_1sig = np.interp(1.0, dchi2[:imin+1][::-1], theta1_grid[:imin+1][::-1])   
# right crossing 
hi_1sig = np.interp(1.0, dchi2[imin:],          theta1_grid[imin:])          

# best-fit central value: fit a parabola to the profile and take its vertex (its minimum).
coef     = np.polyfit(theta1_grid, dchi2, 2)   # a*x^2 + b*x + c
prof_cen = -coef[1] / (2.0 * coef[0])          # vertex = -b / 2a

# 1-sigma errors measured from the best fit, kept separate so they can be asymmetric
err_lo = prof_cen - lo_1sig
err_hi = hi_1sig - prof_cen

print(f"profile best-fit theta_1 = {prof_cen:.2f}  +{err_hi:.2f}  -{err_lo:.2f}")
print(f"profile 1sigma interval  = [{lo_1sig:.2f}, {hi_1sig:.2f}]")
print(f"MCMC posterior mean      = {post1.mean():.2f} +/- {post1.std():.2f}")

---
## Section 9 — Example 2: an informative prior, where they diverge

Two routes lead a posterior to disagree with a profile likelihood:

An informative **explicit prior** is one route — that is this example. 

A second, subtler route is the **prior-volume effect**: marginalising over nuisance parameters with flat priors, across a curved or degenerate likelihood, still re-weights the parameter of interest by how much nuisance volume each value opens up — an "effective prior" you never wrote down. We meet that in Example 3.

Here we keep the **same likelihood** and change **only the prior** on $\theta_1$ to an informative Gaussian. The profile likelihood is unchanged — it follows the *likelihood* and ignores the prior. The posterior, however, is **likelihood × prior**, so its peak is pulled towards the prior. Write the prior and posterior, then run it.

In [ ]:
# --- STUDENT: an informative prior on theta_1, and the resulting posterior ---
MU_PI, SIG_PI = 1.0, 0.8      # informative Gaussian prior on theta_1 (tweak me)

def log_prior_informative(theta):
    """
    Log of a Gaussian prior on theta_1 ~ N(MU_PI, SIG_PI^2); flat on theta_2.
    An additive constant does not matter. Remember this is a LOG prior. 
    """
    pass   

def log_posterior_ex2(theta):
    """
    As always, posterior = likelihood * prior, so in log space:
    log-posterior = log_likelihood_toy(theta) + log_prior_informative(theta).
    """
    pass   # 1 line

The **profile** is identical to Example 1: it uses the **likelihood**, so it ignores the prior. The **poterior** however relies on a prior, here the log_posterior_ex2 and is pulled toward this prior.

Note that we are reusing the profile we caluclated above in the cell below, because nothing has changed on that front. 

In [ ]:
# --- PROVIDED: Example 2 -- profile (likelihood) vs posterior (likelihood x prior) ---

np.random.seed(3)
chain2, acc2 = run_mh_chain(log_posterior_ex2, np.array([2.0, 0.0]),
                            step_size=1.2, n_steps=20000)
post2 = chain2[4000:, 0]

# analytic posterior = product of N(3, Sigma_11=1) [profile] and N(MU_PI, SIG_PI^2) [prior]
var_post  = 1.0 / (1.0/COV_TOY[0, 0] + 1.0/SIG_PI**2)
mean_post = var_post * (MU_TOY[0]/COV_TOY[0, 0] + MU_PI/SIG_PI**2)

prior_curve   = np.exp(-0.5 * ((xs_fine - MU_PI)/SIG_PI)**2)
product_curve = np.exp(-0.5 * (xs_fine - mean_post)**2 / var_post)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

# left: the prior we added
ax[0].plot(xs_fine, prior_curve, color='seagreen', lw=2, ls='--')
ax[0].axvline(MU_PI, color='seagreen', ls=':', lw=1)
ax[0].set_xlabel(PARAM_LABELS[0]); ax[0].set_ylabel('prior density (peak-scaled)')
ax[0].set_title(f'The prior we added:  $\\mathcal{{N}}({MU_PI:g},\\,{SIG_PI:g}^2)$')

# right: profile follows the likelihood; posterior = likelihood x prior
ax[1].plot(theta1_grid, np.exp(-dchi2/2), 'o-', color='steelblue', label='profile likelihood')
ax[1].plot(xs_fine, prior_curve, color='seagreen', ls='--', lw=1.5, label='prior')
ax[1].hist(post2, bins=40, density=True, color='0.8', label='MCMC posterior')
ax[1].plot(xs_fine, product_curve/np.trapz(product_curve, xs_fine), 'm:', lw=1.5,
           label='analytic likelihood$\\times$prior')
ax[1].axvline(MU_TOY[0], color='k', ls=':', lw=1)
ax[1].text(MU_TOY[0]+0.05, 0.15, 'true value (likelihood peak)', rotation=90, fontsize=8, va='bottom')
ax[1].axvline(mean_post, color='m', ls=':', lw=1)
ax[1].set_xlabel(PARAM_LABELS[0]); ax[1].set_ylabel('density (peak-scaled)')
ax[1].set_title('Profile stays; posterior is pulled'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"profile peak (likelihood) : {MU_TOY[0]:.2f}")
print(f"posterior peak (MCMC)     : {post2.mean():.2f}   (analytic {mean_post:.2f})")

---
## Section 10 — Example 3: flat priors, but a hidden prior-volume effect

Now a case where the priors are **flat** — yet the posterior *still* disagrees with the profile. The likelihood below has a nuisance direction $\varphi$ whose good-fit width $w(f)$ **grows** with the parameter of interest $f$.

- **Maximising** (profile) only cares about the *best* $\varphi$ at each $f$ — it never sees how wide the good-fit region is.
- **Integrating** (posterior) sums over all $\varphi$, so values of $f$ that open up more $\varphi$-room collect more posterior mass.

That extra weighting is the **prior-volume effect**: an effective prior that appears from marginalisation even though you wrote down flat priors. 

There is no $1/w(f)$ normalisation here: this width lives in *parameter* space — how tightly the data pin $\varphi$ at each $f$ — and a likelihood is normalised over *data*, not parameters.

In [ ]:
# --- PROVIDED: a likelihood with a prior-volume effect (flat priors) ---
W0, KK, SIG_F = 1.0, 3.0, 0.5      # nuisance width w(f) = W0 (1 + KK f)
F_MIN, F_MAX  = 0.0, 1.0

def log_likelihood_wedge(theta):
    """
    Black-box likelihood over [f, phi]. Key feature: the range of phi that fits
    the data (its width w(f)) GROWS with f. Maximising ignores that width;
    integrating does not.
    """
    f, phi = float(theta[0]), float(theta[1])
    w = W0 * (1.0 + KK * f)
    return -0.5 * (phi / w)**2 - 0.5 * (f / SIG_F)**2

def log_posterior_wedge(theta):
    """
    The posterior below includes a flat prior on f and phi:
    Flat priors: f in [F_MIN, F_MAX], phi in a wide box.
    It simply checks whether the parameters satisfy this prior
    and returns the log_like if they do. 
    """
    f, phi = float(theta[0]), float(theta[1])
    if not (F_MIN <= f <= F_MAX and -20.0 < phi < 20.0):
        return -np.inf
    return log_likelihood_wedge(theta)

def marginal_analytic(fg):
    # marginal posterior of f: integrating out phi gives the closed-form factor w(f).
    g = (1.0 + KK*fg) * np.exp(-0.5 * (fg/SIG_F)**2)   # width factor x likelihood
    return g / g.max()

def marginal_phi_analytic(pg, n_f=400):
    """
    Marginal posterior of the nuisance phi: integrate the likelihood over f in
    [F_MIN, F_MAX]. Here w(f) sits *inside* the exponential, so -- unlike the
    f-marginal above -- there is no closed form; we integrate numerically over a
    fine f-grid for each phi in pg, then peak-normalise.
    """
    f = np.linspace(F_MIN, F_MAX, n_f)
    w = W0 * (1.0 + KK * f)
    integrand = np.exp(-0.5 * (np.asarray(pg)[:, None] / w[None, :])**2
                       - 0.5 * (f[None, :] / SIG_F)**2)   # shape (len(pg), n_f)
    g = np.trapz(integrand, f, axis=1)                    # integrate over f
    return g / g.max()

In [ ]:
# --- PROVIDED: Example 3 -- profile vs posterior of f ---
f_grid          = np.linspace(F_MIN, F_MAX, 21)
COV_SCALE_WEDGE = np.array([0.3, 1.0])      # rough scales for [f, phi]

# (a) PROFILE the LIKELIHOOD: fix f, optimise phi
np.random.seed(4)
chi2_w, theta_w = profile_likelihood(log_likelihood_wedge, 0, f_grid,
                                     np.array([0.5, 0.0]), LADDER, COV_SCALE_WEDGE)
dchi2_w = chi2_w - np.nanmin(chi2_w)

# (b) POSTERIOR with flat priors
np.random.seed(5)
chain_w, acc_w = run_mh_chain(log_posterior_wedge, np.array([0.3, 0.0]),
                              step_size=np.array([0.1, 0.5]), n_steps=40000)
post_w = chain_w[8000:, 0]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

# left: the widening "wedge" likelihood, with the profile ridge
ff = np.linspace(0, 1, 120); pp = np.linspace(-6, 6, 120)
FF, PP = np.meshgrid(ff, pp)
ZW = np.vectorize(lambda a, b: log_likelihood_wedge([a, b]))(FF, PP)
ax[0].contourf(FF, PP, ZW, levels=20, cmap='Blues')
ax[0].plot(ff, np.zeros_like(ff), 'r--', lw=1.5, label='profile ridge ($\\varphi=0$)')
ax[0].set_xlabel('$f$'); ax[0].set_ylabel(r'$\varphi$ (nuisance)')
ax[0].set_title('Good-fit region widens with $f$'); ax[0].legend(fontsize=8, loc='upper left')

# right: profile (peaks at f=0) vs posterior (pushed to f>0)
fg = np.linspace(0, 1, 300)
ax[1].plot(f_grid, np.exp(-dchi2_w/2), 'o-', color='steelblue', label='profile likelihood')
ax[1].hist(post_w, bins=40, density=True, color='0.8', label='MCMC posterior')
marg = marginal_analytic(fg)
ax[1].plot(fg, marg/np.trapz(marg, fg), 'r--', label='analytic marginal')
ax[1].axvline(0.0, color='k', ls=':', lw=1)
ax[1].set_xlabel('$f$'); ax[1].set_ylabel('density (peak-scaled)')
ax[1].set_title('Flat priors, yet posteriors and profile diverge'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"profile peak f      : {f_grid[np.nanargmin(dchi2_w)]:.2f}   (best fit prefers f = 0)")
print(f"posterior mean f    : {post_w.mean():.2f}   (volume pushes it to f > 0)")

In [ ]:
# --- PROVIDED: Example 3 -- profile vs posterior of phi ---
phi_grid          = np.linspace(-7.0, 7.0, 21)
COV_SCALE_WEDGE = np.array([0.3, 1.0])      # rough scales for [f, phi]
param_index = 1

# (a) PROFILE the LIKELIHOOD: fix phi, optimise f
np.random.seed(4)
chi2_w, theta_w = profile_likelihood(log_likelihood_wedge, param_index, phi_grid,
                                     np.array([0.5, 0.0]), LADDER, COV_SCALE_WEDGE)
dchi2_w = chi2_w - np.nanmin(chi2_w)

# (b) POSTERIOR with flat priors
np.random.seed(5)
chain_w, acc_w = run_mh_chain(log_posterior_wedge, np.array([0.3, 0.0]),
                              step_size=np.array([0.1, 0.5]), n_steps=40000)
post_w = chain_w[8000:, param_index]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

# left: the widening "wedge" likelihood, with the profile ridge
ff = np.linspace(0, 1, 120); pp = np.linspace(-6, 6, 120)
FF, PP = np.meshgrid(ff, pp)
ZW = np.vectorize(lambda a, b: log_likelihood_wedge([a, b]))(FF, PP)
ax[0].contourf(FF, PP, ZW, levels=20, cmap='Blues')
ax[0].plot(ff, np.zeros_like(ff), 'r--', lw=1.5, label='profile ridge ($\\varphi=0$)')
ax[0].set_xlabel('$f$'); ax[0].set_ylabel(r'$\varphi$ (nuisance)')
ax[0].set_title('Good-fit region widens with $f$'); ax[0].legend(fontsize=8, loc='upper left')

# right: profile (peaks at f=0) vs posterior (pushed to f>0)
phi_dense = np.linspace(phi_grid[0], phi_grid[-1], 300)
mphi  = marginal_phi_analytic(phi_dense)
mphi /= np.trapz(mphi, phi_dense)              # normalise the analytic marginal to a density
prof  = np.exp(-dchi2_w / 2)
prof *= np.nanmax(mphi) / np.nanmax(prof)      # scale the profile peak to the analytic peak
ax[1].plot(phi_grid, prof, 'o-', color='steelblue', label='profile likelihood')
ax[1].hist(post_w, bins=40, density=True, color='0.8', label='MCMC posterior')
ax[1].plot(phi_dense, mphi, 'r--', label='analytic marginal')
ax[1].axvline(0.0, color='k', ls=':', lw=1)
ax[1].set_xlabel('$\phi$'); ax[1].set_ylabel('density (peak-scaled)')
ax[1].set_title('$\phi$ posteriors and profile agree'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"profile peak phi      : {phi_grid[np.nanargmin(dchi2_w)]:.2f}   (best fit prefers phi = 0)")
print(f"posterior mean phi    : {post_w.mean():.2f}   (posterior peak at phi ~ 0 , no prior-volume effect)")

---
## Section 11 — (Optional) Real-data echo: profile a cosmological parameter

*Scaffold for after class.* We reuse the mock Type Ia supernova data from notebook 1 and profile $H_0$ over the nuisance $M$.

> **Expect a flat profile.** In this low-redshift model $H_0$ and $M$ are perfectly degenerate (only the combination $5\log_{10}(c/H_0)+M$ is constrained), so every $H_0$ fits equally well with a compensating $M$. A flat profile is itself the lesson: the data alone cannot pin $H_0$ without a prior on $M$ or independent data. (A future 2-D model with a weaker degeneracy will make this profile informative.)

In [ ]:
# --- PROVIDED: the SN Ia setup from notebook 1 ---
C_LIGHT = 2.998e5
H0_true, M_true = 70.0, -19.3
np.random.seed(42)
z_obs  = np.sort(np.random.uniform(0.02, 0.15, 25))
mu_obs = 5*np.log10(C_LIGHT*z_obs/H0_true) + 25 + M_true + np.random.normal(0, 0.15, 25)
data   = dict(z=z_obs, mu_obs=mu_obs, mu_err=np.full(25, 0.15),
              H0_true=H0_true, M_true=M_true)

def log_likelihood_sne(theta, data):
    H0 = float(theta[0]); M = float(theta[1]) if len(theta) > 1 else -19.3
    if H0 <= 0:
        return -np.inf
    mu_model = 5*np.log10(C_LIGHT*data['z']/H0) + 25 + M
    return -0.5 * float(np.sum(((data['mu_obs'] - mu_model)/data['mu_err'])**2))

def log_prior_cosmo(theta):
    H0 = float(theta[0]); M = float(theta[1]) if len(theta) > 1 else -19.3
    if 20.0 < H0 < 150.0 and -22.0 < M < -17.0:
        return 0.0
    return -np.inf

In [ ]:
# --- STUDENT (OPTIONAL): profile H0 over M, then plot Delta chi^2 vs H0 ---
H0_grid       = np.linspace(60, 80, 21)
COV_SCALE_SNE = np.array([5.0, 0.1])      # rough scales for [H0, M]

np.random.seed(6)
# TODO: chi2_H0, theta_H0 = profile_likelihood(
#           log_likelihood_sne, 0, H0_grid, np.array([65.0, -19.0]),
#           LADDER, COV_SCALE_SNE, data=data)
# TODO: dchi2_H0 = chi2_H0 - np.nanmin(chi2_H0)
# TODO: plot dchi2_H0 vs H0_grid; mark H0_true; note the flatness (degeneracy)

---
## Section 12 — Profile a real cosmological parameter: the LCDM CMB spectrum

Section 11 profiled a *toy* supernova $H_0$. Here we profile a parameter of the
**real LCDM CMB temperature (TT) likelihood** — the same two-fluid solver you met
in the CMB class and used in notebook 1, Section 7.

Nothing about profiling changes: we reuse `profile_likelihood` exactly. The only
new ingredient is that the nuisance parameters are strongly **correlated**, so
the optimiser at each grid point must step with the full parameter **covariance**
— the very upgrade you gave `propose` in notebook 1, now living inside
`run_annealing`.

We profile the six LCDM parameters
$\theta = [A_s,\ \omega_{\rm cdm},\ \omega_b,\ H_0,\ n_s,\ \Delta N_{\rm eff}]$, with the
default being $H_0$. **You can profile any of them by changing a single index.**

> Flat priors here, so we expect the profile to *agree* with the Bayesian
> posterior marginal (as in Section 7) — a reassuring contrast with the
> divergences of Sections 8 and 9, now on a genuine cosmological model.
> The profile below takes a few minutes; see the setup cell for the Colab
> setup.

#### 12.1 Connecting to the solver

Same setup as notebook 1, Section 7: the cell below clones the CMB class's
[`simplified_CMB_lite`](https://github.com/tsmith2/GGI_cosmology_notebooks)
package, builds its C++ solver, and imports the ready-made likelihood -- so it
works even if you never ran the CMB-class notebooks.

In [ ]:
# --- PROVIDED: get the CMB solver (from the CMB class) and import its likelihood ---
# Self-contained: clones the repo, (re)builds the C++ solver for THIS machine, and
# imports a ready-made likelihood -- so it works even if you never ran the CMB-class
# notebooks. On Colab it clones to /content; locally, next to your working directory.
import sys, os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/tsmith2/GGI_cosmology_notebooks.git'
IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/GGI_cosmology_notebooks') if IN_COLAB else (Path.cwd() / 'GGI_cosmology_notebooks')
PACKAGE_DIR = REPO_DIR / 'simplified_CMB_lite'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
subprocess.run(['make', '-C', str(PACKAGE_DIR / 'cpp'), 'clean'], check=True)   # avoid stale binaries
subprocess.run(['make', '-C', str(PACKAGE_DIR / 'cpp')], check=True)

sys.path.insert(0, str(PACKAGE_DIR / 'python'))
sys.modules.pop('simplified_cmb_lite_colab', None)     # force a fresh import of the downloaded file
from simplified_cmb_lite_colab import (
    load_mock_data, log_prior_lcdm, log_likelihood as log_likelihood_lcdm,
    negative_log_likelihood, tt_spectrum, plot_power_spectrum,
    FID_THETA, LCDM_BOUNDS, PARAM_NAMES,
)
print('CMB solver ready -- theta = [A_s, omega_cdm, omega_b, H0, n_s]')

In [ ]:
# --- PROVIDED: mock data, metadata, and the LCDM log-posterior ---
# log_posterior_lcdm takes no data arg -- it closes over the module global `data`,
# like log_likelihood_toy earlier. The prior and likelihood are imported above.
import numpy as np

data = load_mock_data()
LCDM_NAMES  = ['A_s', 'omega_cdm', 'omega_b', 'H0', 'n_s', 'Delta_Neff']
LCDM_LABELS = PARAM_NAMES
LCDM_FID    = FID_THETA.copy()
LCDM_NDIM   = len(LCDM_FID)

def log_posterior_lcdm(theta):
    """Flat-box prior + Gaussian log-likelihood. Flat prior => this equals the
    likelihood inside the box, so profiling it profiles the likelihood."""
    lp = log_prior_lcdm(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_lcdm(theta, data)

_ndof = len(data['low_ell']) + len(data['bin_center'])
print(f'{_ndof} data points; chi2(fiducial)/dof = {-2 * log_posterior_lcdm(LCDM_FID) / _ndof:.2f}')

#### 12.2 The optimiser, with a covariance

`profile_likelihood`, `run_annealing`, and `annealed_step` are exactly the ones
you built in Sections 4 and 6. Three of them gain the **same 2-D branch** you
added to `propose` in notebook 1: when the step scale is a *matrix* it is treated
as a covariance and proposals are drawn as $\mathcal{N}(0,\ \delta^2 C)$, moving
*along* the parameter degeneracies. A scalar/1-D scale keeps the old meaning, so
every toy example above still runs. (`acceptance_probability`, `annealed_step`,
and `clamp_param` are unchanged and reused as-is.)

In [ ]:
# --- PROVIDED: propose + run_annealing, upgraded with the covariance branch ---
def propose(theta_current, step_size):
    """Gaussian proposal. 2-D step_size -> covariance (multivariate_normal);
    scalar/1-D -> per-parameter widths (the notebook-1 upgrade)."""
    if np.ndim(step_size) == 2:
        return np.random.multivariate_normal(theta_current, step_size)
    return np.random.normal(theta_current, step_size)

def run_annealing(log_post_fn, theta_init, ladder, cov_scale=None, progress=False, **kwargs):
    """Section 4 optimiser, covariance-aware: if cov_scale is a matrix C the step
    is delta**2 * C (a covariance); if it is 1-D the step is delta * cov_scale.

    progress=True shows a per-step bar for THIS single minimisation. It clears
    itself when done (leave=False), so it nests cleanly under the profile bar."""
    theta = np.atleast_1d(np.asarray(theta_init, dtype=float))
    cov_scale = np.ones(len(theta)) if cov_scale is None else np.asarray(cov_scale, float)
    is_cov = cov_scale.ndim == 2
    logpost = log_post_fn(theta, **kwargs)
    theta_best, logpost_best = theta.copy(), logpost
    trajectory, logpost_trace, rungs, step = [theta.copy()], [logpost], [], 0
    bar = (tqdm(total=sum(n for _, _, n in ladder), desc='  minimising', unit='step',
                leave=False) if progress else None)
    for (temperature, delta, n_steps) in ladder:
        step_size = delta**2 * cov_scale if is_cov else delta * cov_scale
        start = step
        for _ in range(n_steps):
            theta, logpost, _ = annealed_step(theta, log_post_fn, step_size, temperature, **kwargs)
            if logpost > logpost_best:
                theta_best, logpost_best = theta.copy(), logpost
            trajectory.append(theta.copy()); logpost_trace.append(logpost); step += 1
            if bar is not None:
                bar.update(1)
        rungs.append((start, step, temperature, delta))
    if bar is not None:
        bar.close()
    return dict(theta_best=theta_best, chi2_best=-2.0 * logpost_best,
                trajectory=np.array(trajectory), chi2_trace=-2.0 * np.array(logpost_trace),
                rungs=rungs)

In [ ]:
# --- PROVIDED: profile_likelihood, covariance-aware (reduce C by the fixed row/col) ---
def profile_likelihood(log_like_fn, param_index, param_grid, theta_init_full,
                       ladder, cov_scale, **kwargs):
    """Section 6 profiler. If cov_scale is a 2-D covariance, the reduced scale is
    C with the fixed parameter's row AND column removed; if 1-D, just delete the
    element (the original behaviour).

    Two nested progress bars: an OUTER one over the grid ('profile'), and an
    INNER per-grid-point one for each minimisation (from run_annealing,
    progress=True). The global fit shows the same inner bar first."""
    n_params = len(theta_init_full)
    cov_scale = np.asarray(cov_scale, float)
    if cov_scale.ndim == 2:
        reduced_scale = np.delete(np.delete(cov_scale, param_index, axis=0), param_index, axis=1)
    else:
        reduced_scale = np.delete(cov_scale, param_index)

    glob = run_annealing(log_like_fn, theta_init_full, ladder, cov_scale=cov_scale,
                         progress=True, **kwargs)
    start_reduced = np.delete(glob['theta_best'], param_index)

    chi2_profile  = np.full(len(param_grid), np.nan)
    theta_profile = np.full((len(param_grid), n_params), np.nan)
    for i, value in enumerate(tqdm(param_grid, desc='profile', unit='pt')):
        reduced_fn = clamp_param(log_like_fn, param_index, value)
        res = run_annealing(reduced_fn, start_reduced, ladder, cov_scale=reduced_scale,
                            progress=True, **kwargs)
        chi2_profile[i]  = res['chi2_best']
        theta_profile[i] = np.insert(res['theta_best'], param_index, value)
        start_reduced    = res['theta_best']
    return chi2_profile, theta_profile

In [ ]:
# --- PROVIDED: the parameter covariance C and the cooling ladder ---
# C was estimated from a short MCMC pilot (as in notebook 1) and hardcoded here so
# this section runs without a pilot chain. Note the enormous dynamic range:
# sqrt(C[0,0]) ~ 3e-12 for A_s vs sqrt(C[3,3]) ~ 0.1 for H0.
C_LCDM = np.array([[1.16707e-23, 9.20929e-16, -8.2717e-17, 2.454e-13, -2.45258e-15, 5.80183e-14], [9.20929e-16, 2.65067e-07, -7.71295e-09, 2.33741e-05, -2.69989e-07, 1.23926e-05], [-8.2717e-17, -7.71295e-09, 1.31661e-09, -8.15653e-07, 1.20271e-08, -4.90098e-07], [2.454e-13, 2.33741e-05, -8.15653e-07, 0.0130637, -6.95408e-05, 0.00184077], [-2.45258e-15, -2.69989e-07, 1.20271e-08, -6.95408e-05, 1.35892e-06, -1.44966e-05], [5.80183e-14, 1.23926e-05, -4.90098e-07, 0.00184077, -1.44966e-05, 0.000666436]])

# a short ladder keeps the profile to a few minutes; raise the step counts for a
# smoother profile (see the note at the end of the section).
LADDER_LCDM = [(1.0, 1.0, 90), (0.3, 0.5, 90), (0.05, 0.25, 120)]

print('per-parameter widths sqrt(diag C):', np.sqrt(np.diag(C_LCDM)))

#### 12.3 Choose a parameter and profile it

Pick which parameter to profile with `PROFILE_INDEX`, then build a grid around
its best fit spanning a few $\sigma$ (so the profile brackets the
$\Delta\chi^2 = 1$ crossings on both sides). The default profiles $H_0$.

In [ ]:
# --- STUDENT: choose the parameter to profile and build its grid ---
PROFILE_INDEX = 3     # 0=A_s  1=omega_cdm  2=omega_b  3=H0  4=n_s  5=Delta_Neff   <-- change me

sigma_j = np.sqrt(np.diag(C_LCDM))[PROFILE_INDEX]     # ~ posterior width of that parameter

# Build ~7-9 grid values spanning roughly the best fit +/- ~4 sigma_j. A quick way
# to centre it near the best fit is a single global anneal:
glob = run_annealing(log_posterior_lcdm, LCDM_FID, LADDER_LCDM, cov_scale=C_LCDM, progress=True)
center = glob['theta_best'][PROFILE_INDEX]
param_grid = None     # e.g. np.linspace(center - 4*sigma_j, center + 4*sigma_j, 7)

print(f'profiling {LCDM_NAMES[PROFILE_INDEX]}: center ~ {center:.4f}, sigma ~ {sigma_j:.4f}')

In [ ]:
# --- STUDENT: run the profile ---
np.random.seed(1)
chi2_prof, theta_prof = None, None    # profile_likelihood(log_posterior_lcdm, PROFILE_INDEX,
                                      #   param_grid, LCDM_FID, LADDER_LCDM, C_LCDM)

dchi2 = None                          # chi2_prof - np.nanmin(chi2_prof)

In [ ]:
# --- PROVIDED: the Bayesian posterior for the SAME parameter (flat prior) ---
# A standalone MCMC chain (~2 min) with the covariance proposal from notebook 1.
np.random.seed(2)
jumping_factor = 2.38 / np.sqrt(LCDM_NDIM)
N_CHAIN = 1500                        # raise for a smoother histogram
chain_lcdm, acc_lcdm = run_mh_chain(log_posterior_lcdm, LCDM_FID,
                                    jumping_factor**2 * C_LCDM, N_CHAIN)
post_param = chain_lcdm[300:, PROFILE_INDEX]
print(f'MCMC acceptance = {acc_lcdm:.2f};  posterior '
      f'{LCDM_NAMES[PROFILE_INDEX]} = {post_param.mean():.4f} +/- {post_param.std():.4f}')

In [ ]:
# --- STUDENT: read the 1-sigma interval off the profile (Delta chi^2 = 1) ---
# This is the same interpolation you used in Section 7: dchi2 is monotonic on each
# side of the minimum, so interpolate the grid value where each branch crosses 1.
imin    = int(np.nanargmin(dchi2))
lo_1sig = None    # np.interp(1.0, dchi2[:imin+1][::-1], param_grid[:imin+1][::-1])
hi_1sig = None    # np.interp(1.0, dchi2[imin:],          param_grid[imin:])

# central value from the parabola vertex; keep the errors separate (can be asymmetric)
coef     = np.polyfit(param_grid, dchi2, 2)
prof_cen = -coef[1] / (2.0 * coef[0])
err_lo, err_hi = None, None    # prof_cen - lo_1sig,   hi_1sig - prof_cen

print(f'profile  {LCDM_NAMES[PROFILE_INDEX]} = {prof_cen:.4f}  +{err_hi:.4f} -{err_lo:.4f}')
print(f'MCMC     {LCDM_NAMES[PROFILE_INDEX]} = {post_param.mean():.4f} +/- {post_param.std():.4f}')

In [ ]:
# --- PROVIDED: profile likelihood vs MCMC posterior, and the Delta chi^2 curve ---
lab = LCDM_LABELS[PROFILE_INDEX]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

# (left) profile likelihood exp(-dchi2/2), peak-scaled onto the posterior histogram
counts, edges = np.histogram(post_param, bins=30, density=True)
prof_like = np.exp(-dchi2 / 2.0)
prof_like = prof_like * counts.max() / np.nanmax(prof_like)
ax[0].plot(param_grid, prof_like, 'o-', color='steelblue', label='profile likelihood')
ax[0].hist(post_param, bins=30, density=True, color='0.8', label='MCMC posterior')
ax[0].axvline(LCDM_FID[PROFILE_INDEX], color='k', ls=':', lw=1, label='input truth')
ax[0].set_xlabel(lab); ax[0].set_ylabel('density (peak-scaled)')
ax[0].set_title('Profile vs posterior'); ax[0].legend(fontsize=8, frameon=False)

# (right) Delta chi^2 with the 1- and 2-sigma levels
ax[1].plot(param_grid, dchi2, 'o-', color='steelblue')
for lvl, name in [(1, r'$1\sigma$'), (4, r'$2\sigma$')]:
    ax[1].axhline(lvl, color='firebrick', ls=':', lw=1)
    ax[1].text(param_grid[-1], lvl + 0.05, name, color='firebrick', ha='right', fontsize=9)
ax[1].axvline(prof_cen, color='steelblue', ls='--', lw=1, label='profile best fit')
ax[1].axvline(LCDM_FID[PROFILE_INDEX], color='k', ls=':', lw=1, label='input truth')
ax[1].set_xlabel(lab); ax[1].set_ylabel(r'$\Delta\chi^2$')
ax[1].set_title(r'Profile $\Delta\chi^2$'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

#### 12.4 What you should see

For a well-constrained parameter under flat priors, the **profile likelihood and
the Bayesian posterior agree**: the $\Delta\chi^2 = 1$ half-width matches the MCMC
standard deviation, and both sit on the input truth. This is the Section 7 lesson,
now on a real CMB likelihood — reassuring, and the baseline against which the
*disagreements* of Sections 8 and 9 stand out.

**Adapt it.** Change `PROFILE_INDEX` to profile any of the six parameters; each
reuses the same covariance and machinery. Some parameters are more correlated
with the others than $H_0$ is — the covariance proposal is what keeps those
profiles efficient.

**Runtime knobs.** Each grid point runs a full annealing optimisation, so cost
scales with the grid size and the ladder length. For a quicker, coarser profile
lower the `n_steps` in `LADDER_LCDM` or shrink the grid; for a smoother one raise
them (and `N_CHAIN` for the histogram). The current settings take roughly
6-7 minutes for the profile plus ~2 for the chain.

This is essentially what [Procoli](https://github.com/tkarwal/procoli) does with a
real Einstein-Boltzmann code in place of the two-fluid solver.

---
## Wrap-up

You started from your notebook-1 Metropolis sampler and, with only small edits, built:

- a **simulated-annealing optimiser** (`run_annealing`) — temperature plus a cooling ladder;
- a **sequential profile-likelihood** routine (`profile_likelihood`) — fix, optimise, warm-start, repeat.

You also saw the central statistical message in three pictures: profiles and posteriors **agree** under flat priors (Example 1), but **diverge** when an explicit prior is informative (Example 2) or when marginalisation introduces a **prior-volume** weighting (Example 3). That is precisely why frequentist profile likelihoods are a useful complement to Bayesian posteriors in cosmology — and the engine you just wrote is the core of [Procoli](https://github.com/tkarwal/procoli).

**Things to explore:** cool too fast (Section 10); widen the volume effect by raising `KK` in Example 3; recover Example 1 from Example 2 by widening `SIG_PI`; tighten the profile grid and relate the spacing to the recommendation $\delta \le \Delta\theta_i/\sigma_i$.